In [5]:
import pandas as pd
import altair as alt

# 1. Carica e prepara i dati
df = pd.read_csv('index-of-cereal-production-yield-and-land-use.csv')
df_world = df[df['Entity'] == 'World'].copy()

# Rinomina le colonne
df_world = df_world.rename(columns={
    'Cereals | 00001717 || Area harvested | 005312 || hectares': 'Area Harvested',
    'Cereals | 00001717 || Production | 005510 || tonnes': 'Production',
    'Cereals | 00001717 || Yield | 005412 || tonnes per hectare': 'Yield',
    'Population (historical)': 'Population'
})

metric_cols = ['Area Harvested', 'Production', 'Yield', 'Population']
BASE_YEAR = df_world['Year'].min() # 1961

# --- INIZIO DELLE MODIFICHE ---

# 2. Calcola la variazione percentuale Anno su Anno (YOY)
# Rimuovi l'indice 1961=100 se non ti serve più, altrimenti esegui entrambi i calcoli.

for col in metric_cols:
    # Calcola la variazione percentuale (ad esempio, 0.05 per +5%)
    df_world[f'{col} YOY Change'] = df_world[col].pct_change() * 100 
    
    # Se vuoi anche mantenere il calcolo dell'indice base:
    df_base = df_world[df_world['Year'] == BASE_YEAR]
    base_values = {col: df_base[col].iloc[0] for col in metric_cols}
    df_world[f'{col} Index (1961=100)'] = (df_world[col] / base_values[col]) * 100

# --- FINE DELLE MODIFICHE ---

# 3. Aggiorna la selezione delle colonne
# Includi le nuove colonne YOY per salvarle o visualizzarle
yoy_cols = [f'{col} YOY Change' for col in metric_cols]
index_cols = ['Year'] + [f'{col} Index (1961=100)' for col in metric_cols]

# Seleziona tutte le colonne originali + Indici + YOY per il salvataggio
all_new_cols = ['Entity'] + ['Year'] + metric_cols + index_cols[1:] + yoy_cols

# 4. Salva il DataFrame aggiornato con le nuove colonne (Opzionale)
output_csv_filename = 'indexed_and_yoy_cereal_data.csv'
df_world.to_csv(output_csv_filename, index=False, columns=all_new_cols)

print(f"File salvato con successo: {output_csv_filename}")
print(f"Le nuove colonne YOY aggiunte sono: {yoy_cols}")

# --- RESTO DEL CODICE PER LA VISUALIZZAZIONE ALTAIR ---

# 5. Prepara i dati per la visualizzazione Altair (usa i dati dell'Indice come prima)
df_indexed = df_world[index_cols]
df_long = df_indexed.melt(
    id_vars=['Year'],
    value_vars=[col for col in index_cols if col != 'Year'],
    var_name='Metric',
    value_name='Index Value'
)

# 6. Crea la visualizzazione Altair (line graph)
chart = alt.Chart(df_long).mark_line(point=True).encode(
    x=alt.X('Year:Q', axis=alt.Axis(format='d', title='Anno')),
    y=alt.Y('Index Value:Q', title='Indice di Crescita (1961 = 100)'),
    color=alt.Color('Metric:N', title='Metrica'),
    tooltip=[
        alt.Tooltip('Year:Q', format='d', title='Anno'),
        alt.Tooltip('Metric:N', title='Metrica'),
        alt.Tooltip('Index Value:Q', format=',.1f', title='Indice (1961=100)')
    ]
).properties(
    title='Indici di Crescita Globale (Cereali e Popolazione, 1961=100)'
).interactive()

# 7. Salva il grafico interattivo
output_filename = 'global_index_linechart.json'
chart.save(output_filename)

File salvato con successo: indexed_and_yoy_cereal_data.csv
Le nuove colonne YOY aggiunte sono: ['Area Harvested YOY Change', 'Production YOY Change', 'Yield YOY Change', 'Population YOY Change']


C:\Users\anna_\Desktop\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


In [ ]:
import pandas as pd
import altair as alt

# 1. Carica e prepara i dati
df = pd.read_csv('index-of-cereal-production-yield-and-land-use.csv')
df_world = df[df['Entity'] == 'World'].copy()

# Rinomina le colonne originali
df_world = df_world.rename(columns={
    'Cereals | 00001717 || Area harvested | 005312 || hectares': 'Area Harvested',
    'Cereals | 00001717 || Production | 005510 || tonnes': 'Production',
    'Cereals | 00001717 || Yield | 005412 || tonnes per hectare': 'Yield',
    'Population (historical)': 'Population'
})

metric_cols = ['Area Harvested', 'Production', 'Yield', 'Population']
BASE_YEAR = df_world['Year'].min() # 1961

# 2. Calcola Indici e Variazioni Percentuali
df_base = df_world[df_world['Year'] == BASE_YEAR]
base_values = {col: df_base[col].iloc[0] for col in metric_cols}

yoy_cols = []
shifted_index_cols = []
index_100_cols = []

for col in metric_cols:
    # a. Calcola l'Indice Standard (1961=100)
    index_100_name = f'{col} Index (1961=100)'
    df_world[index_100_name] = (df_world[col] / base_values[col]) * 100
    index_100_cols.append(index_100_name)

    # b. Calcola l'Indice Spostato (1961=0)
    shifted_index_name = f'{col} Index (1961=0) - Net Change (%)'
    df_world[shifted_index_name] = df_world[index_100_name] - 100
    shifted_index_cols.append(shifted_index_name)
    
    # c. Calcola la Variazione Percentuale Anno su Anno (YOY)
    yoy_name = f'{col} YOY Change (%)'
    df_world[yoy_name] = df_world[col].pct_change() * 100
    yoy_cols.append(yoy_name)


# 3. Salva il DataFrame completo in un nuovo file CSV
# Definisce l'ordine delle colonne nel file di output
output_columns = (
    ['Entity', 'Year'] + 
    metric_cols + 
    index_100_cols + 
    shifted_index_cols + 
    yoy_cols
)

output_csv_filename = 'global_data_elaborato.csv'
df_world.to_csv(output_csv_filename, index=False, columns=output_columns)

print(f"File salvato con successo: {output_csv_filename}")
print(f"Le nuove colonne aggiunte sono:")
print(shifted_index_cols)
print(yoy_cols)


# 4. Prepara i dati per la visualizzazione Altair (usa l'Indice Spostato)
df_indexed = df_world[['Year'] + shifted_index_cols]

df_long = df_indexed.melt(
    id_vars=['Year'],
    value_vars=shifted_index_cols,
    var_name='Metric',
    value_name='Growth Net Change (%)'
)

# 5. Crea la visualizzazione Altair (line graph)
chart = alt.Chart(df_long).mark_line(point=True).encode(
    x=alt.X('Year:Q', axis=alt.Axis(format='d', title='Anno')),
    y=alt.Y('Growth Net Change (%):Q', 
            title='Variazione Netta (%) dal 1961 (1961 = 0)',
            scale=alt.Scale(zero=True) 
           ),
    color=alt.Color('Metric:N', title='Metrica'),
    tooltip=[
        alt.Tooltip('Year:Q', format='d', title='Anno'),
        alt.Tooltip('Metric:N', title='Metrica'),
        alt.Tooltip('Growth Net Change (%):Q', format=',.1f', title='Variazione Netta (%)')
    ]
).properties(
    title='Crescita Cereali e Popolazione: Variazione Percentuale dal 1961'
).interactive()

# 6. Salva il grafico interattivo
output_filename = 'global_shifted_index_linechart.json'
chart.save(output_filename)

In [7]:
import pandas as pd
import altair as alt

# 1. Carica e prepara i dati
df = pd.read_csv('index-of-cereal-production-yield-and-land-use.csv')
df_world = df[df['Entity'] == 'World'].copy()

# Rinomina le colonne originali
df_world = df_world.rename(columns={
    'Cereals | 00001717 || Area harvested | 005312 || hectares': 'Area Harvested',
    'Cereals | 00001717 || Production | 005510 || tonnes': 'Production',
    'Cereals | 00001717 || Yield | 005412 || tonnes per hectare': 'Yield',
    'Population (historical)': 'Population'
})

metric_cols = ['Area Harvested', 'Production', 'Yield', 'Population']
BASE_YEAR = df_world['Year'].min() # 1961

# 2. Calcola Indici e Variazioni Percentuali
df_base = df_world[df_world['Year'] == BASE_YEAR]
base_values = {col: df_base[col].iloc[0] for col in metric_cols}

yoy_cols = []
shifted_index_cols = []
index_100_cols = []

for col in metric_cols:
    # a. Calcola l'Indice Standard (1961=100)
    index_100_name = f'{col} Index (1961=100)'
    df_world[index_100_name] = (df_world[col] / base_values[col]) * 100
    index_100_cols.append(index_100_name)

    # b. Calcola l'Indice Spostato (1961=0)
    shifted_index_name = f'{col} Index (1961=0) - Net Change (%)'
    df_world[shifted_index_name] = df_world[index_100_name] - 100
    shifted_index_cols.append(shifted_index_name)
    
    # c. Calcola la Variazione Percentuale Anno su Anno (YOY)
    yoy_name = f'{col} YOY Change (%)'
    df_world[yoy_name] = df_world[col].pct_change() * 100
    yoy_cols.append(yoy_name)


# 3. Salva il DataFrame completo in un nuovo file CSV
# Definisce l'ordine delle colonne nel file di output
output_columns = (
    ['Entity', 'Year'] + 
    metric_cols + 
    index_100_cols + 
    shifted_index_cols + 
    yoy_cols
)

output_csv_filename = 'global_data_elaborato.csv'
df_world.to_csv(output_csv_filename, index=False, columns=output_columns)

print(f"File salvato con successo: {output_csv_filename}")
print(f"Le nuove colonne aggiunte sono:")
print(shifted_index_cols)
print(yoy_cols)



File salvato con successo: global_data_elaborato.csv
Le nuove colonne aggiunte sono:
['Area Harvested Index (1961=0) - Net Change (%)', 'Production Index (1961=0) - Net Change (%)', 'Yield Index (1961=0) - Net Change (%)', 'Population Index (1961=0) - Net Change (%)']
['Area Harvested YOY Change (%)', 'Production YOY Change (%)', 'Yield YOY Change (%)', 'Population YOY Change (%)']
